#### Structured output

##### Models can be requested to provide their response in a format matching a given schema 

##### This is useful for ensuring the output can be easily parsed and used in subsequent processing.

### pydantic

In [3]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [13]:
model=init_chat_model(model="gemini-2.5-flash", model_provider="google_genai")

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="title of the movie")
    name:str = Field(description="")
    price:int = Field(description="Tiket Price of movie")
    rate:float = Field(description="Rating Of moviee")
    

In [6]:
structure_output=model.with_structured_output(Movie)
print(structure_output)

first=_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17', 'langchain-google-genai': '4.3.5'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000267FCB25760>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'ti

In [ ]:
response=model.invoke("Provide Details of sex education movie")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content="Sex education movies are a diverse category of films designed to inform and educate audiences, primarily children, adolescents, and young adults, about various aspects of human sexuality, relationships, and reproductive health. Their content, tone, and target audience can vary significantly.\n\nHere are common details and characteristics you might find in a sex education movie:\n\n1.  **Target Audience:**\n    *   **Children (Elementary School):** Often focus on basic anatomy, body safety, puberty's initial stages (like growth spurts), and healthy friendships. Animation is common.\n    *   **Pre-Teens/Early Teens (Middle School):** Delve deeper into puberty (physical and emotional changes for both sexes), menstruation, basic reproductive anatomy, hygiene, body image, and the concept of consent in a broader sense (personal boundaries).\n    *   **Teens/Young Adults (High School & Beyond):** Cover more complex topics like sexual consent, contraception, sexually transmi

In [8]:
structure_output.invoke("Provide Details of sex education movie")

Movie(title='Sex Education', name='Sex Education', price=15, rate=8.3)

In [ ]:
# Raw values

In [18]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field("...",description="title of the movie")
    name:str = Field("...",description="")
    price:int = Field("...",description="Tiket Price of movie")
    rate:float = Field("...",description="Rating Of moviee")

structured_output=model.with_structured_output(Movie,include_raw=True)
result =structured_output.invoke("Provide Movie details Inception")

result 

{'raw': AIMessage(content='{"title":"Inception","name":"Inception","price":15,"rate":8.8}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a056cf-9e55-7420-9513-108594321298-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 194, 'total_tokens': 200, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 172}}),
 'parsed': Movie(title='Inception', name='Inception', price=15, rate=8.8),
 'parsing_error': None}

#####  Nested Strcuture

In [22]:
class Actor(BaseModel):
    name:str = Field(...,description="Name of Movie Actor")
    role:str = Field(...,description="Role in Movie")

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

structure_output=model.with_structured_output(MovieDetails) # bind the structure
result=structure_output.invoke("Provide Movie Details Himmatwala")
result

MovieDetails(title='Himmatwala', year=2013, cast=[Actor(name='Ajay Devgn', role='Ravi'), Actor(name='Tamannaah Bhatia', role='Rekha'), Actor(name='Paresh Rawal', role='Narayan Das')], genres=['Action', 'Comedy', 'Drama'], budget=11.0)

### TypeDict
##### TypedDict does not perform runtime validation like Pydantic.

In [24]:
from typing_extensions import TypedDict,Annotated

class Movie(TypedDict):
    title : Annotated[str,...,"Movie Title"]
    actor : Annotated[str,...,"Movie Actor"]

structured_op=model.with_structured_output(Movie)
res=structured_op.invoke("provide information of inception movie")
res

{'title': 'Inception', 'actor': 'Leonardo DiCaprio'}

In [25]:
class Actor(TypedDict):
    name:str = Field(...,description="Name of Movie Actor")
    role:str = Field(...,description="Role in Movie")

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

structure_output=model.with_structured_output(MovieDetails) # bind the structure
result=structure_output.invoke("Provide Movie Details Himmatwala")
result


{'title': 'Himmatwala',
 'year': 2013,
 'cast': [{'name': 'Ajay Devgn', 'role': 'Ravi'},
  {'name': 'Tamannaah Bhatia', 'role': 'Rekha'},
  {'name': 'Paresh Rawal', 'role': 'Narayandas'}],
 'genres': ['Action', 'Comedy', 'Drama'],
 'budget': 12000000}

In [26]:
model.profile

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-06-17',
 'last_updated': '2025-06-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

### Data Class
##### A dataclass is a Python class used to store structured data with less code.
##### A data class is a class typically containing mainly data, although there aren’t really any restrictions

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

personinfo(title='...', name='Ravi Nikam', price='...', rate='...')

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'email': 'ravinikam0999@gmail.com',
 'name': 'Ravi Nikam',
 'phone': 9876543210,
 'rate': 55.75}

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]